# 지식과 메모리

* 지식은 기술 스펙, 정책 문서, 상품 카탈로그, 고객 또는 시스템 로그 같은 사실이나 도메인 특화 콘텐츠를 생성 시점에 끌어와 에이전트가 검증 가능한 정보를 알도록 만듦

* 메모리는 에이전트 자신의 히스토리를 포착
* 메모리를 통해 에이전트는 여러 턴과 세션에 걸쳐 연속성을 유지하며 과거 상호작용을 기억하고 이 이력을 활용해 앞으로의 의사결정을 내릴 수 있음

## 메모리 기본 사용법

### 컨텍스트 윈도우 관리

* 가장 단순한 메모리 접근법은 컨텍스트 윈도에 의존하는 것

* 컨텍스트 윈도우
    * 한 번의 호출에서 파운데이션 모델에 입력으로 전달되는 정보
* 컨텍스트 길이
    * 파운데이션 모델이 한 번의 호출에서 입력으로 받아들이고 주의를 기울일 수 있는 최대 토큰 수

* 가장 단순한 방식에서 컨텍스트 윈도우는 현재 질문과 현재 세션에서의 모든 이전 상호작용을 포함
* 윈도우가 가득 차면 가장 최근 상호작용만 포함, 이전 내용은 삭제
* 어떤 상황에서는 컨텍스트 윈도우에 담을 수 있는 것보다 더 많은 정보를 제공해야 하는 경우가 있음  
-> 제한된 토큰 예산을 어떻게 배분할지에 대해 신중해야 함

* 단순한 사용 사례에서는 롤링 컨텍스트 윈도우를 사용할 수 있음  
-> 파운데이션 모델과의 상호작용이 진행되는 동안 전체 대화 내용을 컨텍스트 윈도우에 계속 전달함, 선입선출 방식
* 구현이 쉽고 복잡도가 낮으나 상호작용이 충분히 길어져 현재 컨텍스트에서 밀려나는 순간부터 정보가 사라짐  
-> 관련성 높은 컨텍스트를 강조하고 프롬프트의 뒤에 가깝게 배치하면 사용될 가능성을 높일 수 있음

* 맥락을 유지하고 단기 기억을 구현하는 예시

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, MessagesState, START

# LLM 초기화
llm = init_chat_model(model="gpt-5-mini", temperature=0)

def call_model(state: MessagesState):
    response = llm.invoke(state["messages"])
    return {"messages": response}

builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")
graph = builder.compile()

# 메모리가 없어서 대화 상태를 유지할 수 없음
input_message = {"type": "user", "content": "안녕하세요! 제 이름은 철수입니다."}
for chunk in graph.stream({"messages": [input_message]}, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

input_message = {"type": "user", "content": "제 이름이 뭐라고요?"}
for chunk in graph.stream({"messages": [input_message]}, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

from langgraph.checkpoint.memory import MemorySaver

# 메모리가 유지되어 대화 상태를 유지할 수 있음
memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "1"}}
input_message = {"type": "user", "content": "안녕하세요! 제 이름은 철수입니다."}
for chunk in graph.stream({"messages": [input_message]}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

### 전체 텍스트 검색

* 대규모 검색 및 검색 증강 시스템의 기반을 이루며 파운데이션 모델을 사용하는 에이전트에 정밀한 과거 컨텍스트를 주입하는 견고하고 성숙한 접근법을 제공

* 역색인
    * 모든 텍스트를 전처리하면서 토크나이징, 정규화, 불용어 제거를 수행한 뒤, 그 용어를 그 용어가 등장하는 메시지 청크나 문서 목록에 매핑  
    -> 지정된 모든 메시지를 스캔하지 않고도 빠르게 조회 가능

* 에이전트는 쿼리 키워드를 포함하는 구절만 정확히 가져오기 위해 해당 용어의 포스팅 리스트를 따라가기만 하면 됨
* 검색 결과를 관련도 순으로 정렬하기 위해 대부분의 시스템은 BM25 스코어링 함수를 사용
    * BM25
        * 각 구절을 용어 빈도, 역문서 빈도, 문서 길이 정규화에 따라 가중

* 사용자 쿼리가 들어오면 인덱싱에 사용했던 것과 동일한 텍스트 파이프라인으로 쿼리를 분석하고 BM25가 상위 K개 후보 구절 목록을 점수 순으로 정렬해 제공
* 얻은 상위 결과는 종종 잘라내거나 요약한 뒤 파운데이션 모델 프롬프트에 직접 주입  
-> 컨텍스트 길이를 소진하지 않고도 가장 관련성 높은 과거 컨텍스트를 확인할 수 있음

* 키워드 기반 검색을 수행하는 예시

In [2]:
from rank_bm25 import BM25Okapi
from typing import List

corpus: List[List[str]] = [
    "에이전트 J는 패기가 넘치는 신입 대원이다".split(),
    "에이전트 K는 수년간의 MIB 경험과 멋진 뉴럴라이저를 갖고 있다".split(),
    "두 명의 에이전트가 검은 정장을 입고 은하계를 구했다".split(),
]
# 2. BM25 인덱스 생성
bm25 = BM25Okapi(corpus)

# 3. 간단한 쿼리로 검색 수행
query = "신입 대원은 누구지?".split()
top_n = bm25.get_top_n(query, corpus, n=2)

print("쿼리:", " ".join(query))
print("상위 일치 문장:")
for line in top_n:
    print(" •", " ".join(line))

쿼리: 신입 대원은 누구지?
상위 일치 문장:
 • 에이전트 J는 패기가 넘치는 신입 대원이다
 • 두 명의 에이전트가 검은 정장을 입고 은하계를 구했다


## 시맨틱 메모리와 벡터 스토어

* 시맨틱 메모리는 일반적인 지식, 개념 과거 경험을 저장하고 검색하는 장기 메모리의 한 유형으로 이러한 시스템의 인지 능력을 강화하느 데 중요한 역할을 함  
    -> 정보와 과거 경험을 저장해 두었다가 이후 성능을 향상시키기 위해 필요 시 효율적으로 다시 꺼내 쓸 수 있음

-> 대표적인 방법이 벡터 데이터베이스

### 시맨틱 검색

* 쿼리 뒤에 있는 컨텍스트와 의도를 이해하는 것을 목표로 함
* 시맨틱 검색의 핵심은 정확한 문자열 일치보다는 단어와 구의 의미에 초점을 맞추는 것
* 머신러닝 기법을 활용해 컨텍스트, 동의어, 단어 사이의 관계를 해석  
-> 검색 시스템은 사용자의 의도를 파악하고 정확히 같은 검색어를 포함하지 않더라도 컨텍스트상 관련성이 높은 결과를 제공할 수 있음

* 해당 접근법의 기반은 임베딩
* 임베딩은 대규모 텍스트 코퍼스에서의 사용 맥락을 바탕으로 단어의 의미를 포착한 벡터 표현

* 시맨틱 검색은 특히 정확히 같은 키워드를 공유하지 않는 문서 전반에서 의미적으로 관련된 정보를 거맥하는 측면에서 에이전틱 시슽메 내부 메모리의 성능을 향상시키는 데 매우 유용한 기법임이 입증됨

### 벡터 스토어로 시맨틱 메모리 구현

* 임베딩은 보통 파운데이션 모델이나 기타 자연어 처리 기법을 사용해 텍스트 정보를 조밀한 벡터 표현으로 인코딩하는 방식으로 만들어짐
* 얻은 벡터 표현은 연속적인 벡터 공간에서 데이터 포인트의 시맨틱 속성과 관계를 포착
* 벡터 표현을 얻으면 효율적으로 저장할 장소가 필요함  
-> 벡터 데이터베이스

* 에이전트가 쿼리를 받거나 정보를 검색할 필요가 있을 때 쿼리 임베딩을 기반으로 벡터 스토어에서 유사도 검색을 수행할 수 있음  
-> 조회는 매우 빠르게 수행될 수 있어 대량의 정보 위에서도 고품질의 액션과 응답을 제공하기 위한 효율적인 검색 경로를 마련해 줌

* 구현 예시

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage
from langgraph.graph import StateGraph, MessagesState, START

# LLM 초기화
llm = init_chat_model(model="gpt-5-mini", temperature=0)

def call_model(state: MessagesState):
    context = "\n\n".join([r.get("text", "") for r in results])
    messages = [SystemMessage(content="참고:\n" + context)] + list(state["messages"])
    response = llm.invoke(messages)
    return {"messages": response}


from vectordb import Memory

memory = Memory(chunking_strategy={'mode':'sliding_window', 'window_size': 128, 'overlap': 16})

text = """
머신러닝은 분석 모델 구축을 자동화하는 데이터 분석 방법이다.

이는 시스템이 데이터로부터 학습하고 패턴을 식별하며 최소한의 인간 개입으로 의사결정을 내릴 수 있다는 개념에 기반한 인공지능의 한 분야이다.

머신러닝 알고리즘은 원하는 출력 사례를 포함한 데이터셋으로 학습된다. 예를 들어, 이미지를 분류하는 머신러닝 알고리즘은 고양이와 개의 이미지를 포함한 데이터셋으로 훈련될 수 있다.

알고리즘이 학습을 마치면 새로운 데이터에 대한 예측에 사용될 수 있다. 예를 들어, 이미지 분류 알고리즘은 새로운 이미지에 고양이가 있는지 개인지가 있는지를 예측하는 데 활용될 수 있다.
"""

metadata = {"title": "Introduction to Machine Learning", "url": "https://learn.microsoft.com/en-us/training/modules/introduction-to-machine-learning"}

memory.save(text, metadata)

text2 = """
인공지능(AI)은 인간처럼 사고하고 행동을 모방하도록 프로그래밍된 기계에서 인간 지능을 시뮬레이션하는 것을 의미한다.

이 용어는 학습과 문제 해결과 같이 인간의 정신과 연관된 특성을 보이는 모든 기계에 적용될 수 있다.

AI 연구는 게임 플레이부터 의료 진단에 이르기까지 매우 다양한 문제를 해결하기 위한 효과적인 기법을 개발하는 데 큰 성공을 거두었다.
"""

metadata2 = {"title": "Introduction to Artificial Intelligence", "url": "https://microsoft.github.io/AI-For-Beginners/"}

memory.save(text2, metadata2)

query = "AI와 머신러닝은 어떤 관계가 있나요?"

results = memory.search(query, top_n=3)

builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")
graph = builder.compile()

input_message = {"type": "user", "content": "AI와 머신러닝은 어떤 관계가 있나요?"}
for chunk in graph.stream({"messages": [input_message]}, {}, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

print(results)

### RAG: 검색 증강 생성

* 메모리를 통합한다는 것은 단순히 지식을 저장하고 관리하는 수준을 넘어 시스템이 컨텍스트적으로 적절하고 정확한 응답을 생성하는 능력을 향상시키는 작업을 포함
* RAG
    * 검색 기반 기법과 생성 모델의 장점을 결합해 목표를 달성하는 강력한 기법

* RAG 인덱싱 파이프라인
    1. 시스템이 질문에 답하는 데 도움이 될 수 있는 문서 집합을 준비
    2. 문서들을 더 작은 청크로 나눔
    3. 잘라낸 청크들을 인코더 모델로 임베딩한 뒤 벡터 데이터베이스에 인덱싱

* 검색 단계에서는 시스템이 대규모 문서 코퍼스나 임베딩 벡터 스토어에서 주어진 쿼리나 컨텍스트와 관련된 정보 조각을 찾음
* 생성 단계에서는 검색된 정보가 생성 파운데이션 모델로 전달되고 모델은 이 컨텍스트를 활용해 일관되고 상황에 맞는 응답을 생성

* 외부 지식을 활용해 생성 과정에 통합함으로써 RAG는 더 정보에 기반하고 더 정확하며 컨텍스트적으로 더 적절한 응답을 생성

### 시맨틱 경험 메모리

* 시맨틱 스토어만으로는 에이전트가 매 세션을 백지 상태에서 시작하게 되고 장기 실행되거나 복잡한 작업의 컨텍스트가 점차 컨텍스트 윈도우 밖으로 밀려나 사라지게 됨  
-> 시맨틱 경험 메모리는 모두 완화할 수 있음

* 사용자 입력이 들어올 때마다 텍스트는 임베딩 모델을 사용해 벡터 표현으로 변환
* 얻은 임베딩은 메모리 스토어에 저장된 과거 모든 상호작용을 대상으로 수행하는 벡터 검색의 쿼리로 사용됨
* 컨텍스트 윈도우 일부는 시맨틱 경험 메모리에서 검색된 최적 일치 결과들을 위해 예약되고 나머지 공간은 시스템 메시지, 최신 사용자 입력, 가장 최근 상호작용에 할당
* 시맨틱 경험 메모리를 사용하면 더 적응적이고 개인화된 행동이 가능해짐

## 그래프RAG

* 그래프 기반 데이터 구조를 도입해 검색 과정을 향상시키는 고급 RAG 확장 기법  
-> 생성되는 콘텐츠의 풍부함과 정확성을 크게 높일 수 있음

### 지식 그래프 활용

* 그래프RAG에서 검색 단계는 관련 문서나 스니펫을 가져오는 데 그치지 않고 데이터 안의 복잡한 관계와 컨텍스트를 표현하는 그래프에서 노드와 엣지를 분석하고 검색

* 그래프RAG의 세 가지 구성 요소
    * 지식 그래프
        * 구성 요소는 데이터를 그래프 형식으로 저장하며 엔티티와 이들의 관계가 명시적으로 정의
        * 그래프 데이터베이스는 서로 연결된 데이터를 관리하고 여러 홉이나 다중 관계를 포함하는 복잡한 쿼리를 처리하는 데 매우 효율적
    * 검색 시스템
        * 그래프RAG의 검색 시스템은 그래프 데이터베이스를 효율적으로 질의해 입력 쿼리나 컨텍스트와 가장 관련성 높은 서브그래프나 노드 클러스터를 추출하도록 설계됨
    * 생성 모델
        * 그래프 형태로 검색된 관련 데이터를기반으로 생성 모델은 이 정보를 종합해 일관되고 컨텍스트가 풍부한 응답을 생성

* 그래프RAG에서 지식 그래프를 활용하면 정보를 검색하고 생성에 활용하는 방식이 변하고 다양한 애플리케이션에서 더 지능적이고 컨텍스트에 민감하며 정확한 응답을 가능하게 함

### 지식 그래프 구축

* 지식 그래프는 그래프RAG 시스템을 포함한 지능형 시스템의 역량을 강화하는 구조화되고 시맨틱하게 풍부한 정보를 제공하는 데 핵심적인 역할을 함

* 지식 그래프를 구성하는 프로세스
    1. 데이터 수집
    2. 데이터 전처리
    3. 엔티티 인식 및 추출
    4. 관계 추출
    5. 온톨로지 설계
    6. 그래프 채우기
    7. 통합 및 검증
    8. 유지보수와 업데이트

### 동적 지식 그래프의 가능성과 위험성

* 동적 지식 그래프는 실시간 애플리케이션에서 지식을 관리하고 활용하는 방식에서 큰 진전을 의미

* 최근 모델 아키넽거츠이 발전으로 LLM이 한 번에 문서 전체를 기억하고 처리할 수 있게 됨
* 하지만 검색 없는 접근 방식에는 분명한 트레이드오프가 존재
    * 막대한 연산이 필요하며 지연시간과 비용 측면에서 부담이 커질 수 있음
    * 외부 검색을 제거하면서 기대했던 단순화 효과가 오히려 상쇄될수도 있음
    * 큰 컨텍스트 윈도우 안에서 정말로 관련된 정보를 제대로 집어낸다는 보장이 없음

* 동적 실시간 정보 처리는 실시간 데이터를 통합할 수 있는 동적 지식 그래프 덕분에 큰 폭으로 향상  
-> 동적 지식 그래프는 큰 이점을 제공

* 적응형 학습 역시 동적 그래프의 중요한 특징
* 주기적인 재학습이나 수동 업데이트 없이도 새로운 데이터로부터 계속해서 스스로를 갱신
* 지식 그래프는 연산과 추론에 적합한 구조화된 포맷으로 중요한 정보를 제공하며 벡터 스토어보다 더 높은 유연성을 제공

* 단점
    * 유지보수의 복잡성
    * 자원 집약성
    * 보안 및 프라이버시 우려
    * 의존성과 과도한 신뢰

* 동적 지식 그래프의이점을 최대한 활용하면서도 위험을 줄이기 위해 몇 가지 전략을 함께 사용하는 것이 좋음
    * 자동화 도구와 프로세스를 활용해 강력한 검증 메커니즘을 구축하고 그래프 내부 데이터의 정확성과 신뢰성을 지속적으로 점검해야 함
    * 확장 가능한 아키텍처를 설계함으로써 동적 그래프의 연산 부담을 감당할 수 있어야 함
    * 보안 측면에서는 모든 데이터 입력과 통합이 최신 보안, 프라이버시 규정을 준수하도록 해야 함
    * 중요한 의사결정 과정에는 반드시 사람의 검토를 포함시켜 시스템에 대한 과도한 의존으로 인한 위험을 완화해야 함

### 노트 작성

* 파운데이션 모델이 질문에 바로 답하려 하지 않고 입력 컨텍스트에 대한 노트를 의도적으로 생성하도록 프롬프트
* 노트 작성은 질문이 주어지기 전에 수행, 이후 현재 작업을 처리할 때 노트들을 원래 컨텍스트와 섞어서 사용